# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/syeddaniyalg/flyrank-work/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Abstract** – We built a predictive ranking model to help content editors prioritise pages that are likely to experience a click‑through rate (CTR) decline. Using 3.6 million daily search performance records from the FlyRank internship warehouse, we aggregated impressions and average position over the first 15 days of March 2026 as features, and defined the label as a CTR drop between the first and second halves of the month. A Random Forest classifier, validated on a client‑held‑out split, achieved a Precision@50 of 0.84, compared to a hand‑written baseline rule of 0.52 (base rate ~0.40). The model provides a directional signal for editorial triage and is accompanied by a transparent action playbook with reason codes and human review guidelines. The work is observational and decision‑support; no causal claims are made.

## 1. Question

*The research question and the decision it supports.*

**Research question:** Out of thousands of visible pages, which ones are most likely to under‑perform their ranking position and lose clicks, and should be reviewed first?

**Decision supported:** A content editor with limited capacity decides which page titles, meta descriptions, or snippets to review this week.

**Action:** Human review of the flagged pages; the model provides a ranked queue with reason codes.

**Cost of a wrong call:** A false positive wastes editor time; a false negative misses an opportunity to protect traffic. Both costs are measured in lost opportunity, not ranking loss itself.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

**Source:** FlyRank internship warehouse (Hugging Face).  
**Main table:** `fact_content_daily_performance`, partitioned by month.  
**Development month:** March 2026.  
**Filter:** `gsc_data_available = TRUE` and `gsc_impressions > 0` – reduced to 3.6 million daily rows.  
**Aggregation:** Grouped by `client_hash_id` and `content_hash_id`, summing impressions and clicks, averaging position.  
**Initial frame:** 77,540 content items with at least 100 impressions in the first 15 days.  
**Final modeling frame:** 77,400 items after requiring a matching row in the second 15‑day window (non‑null `ctr_curr`). All reported metrics use this final frame.

**Excluded columns:** GA4 engagement columns (due to three‑valued `ga4_data_available` and zero‑fill before client start date) and AI referral columns (too sparse).

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**Features:** `impressions_prev` and `avg_position_prev` from the first 15 days of March 2026.

**Label:** `is_declining_label = 1` if `ctr_curr < ctr_prev`, where `ctr` is computed from clicks / impressions in each 15‑day window.

**Baseline:** Hand‑written rule:
- `good_tier = position_tier in ["top_3","page_1","striking"]`
- `positive_gap = (tier_avg_ctr - ctr_prev) > 0`
- `score = good_tier * positive_gap * (tier_avg_ctr - ctr_prev) * impressions_prev`

**Model:** Random Forest (300 trees, max_depth=6, class_weight='balanced').

**Validation:** Client‑grouped split (GroupShuffleSplit, 80/20) – no client appears in both train and test.

**Leakage checks:** Adding `ctr_prev` and `ctr_curr` to features inflated Precision@50 to 1.00, confirming these signals are unsafe. Final model uses only the two pre‑declaration features.

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

| K | Baseline (hand rule) Precision@K | Random Forest Precision@K | Base rate |
|---|----------------------------------|---------------------------|-----------|
| 20 | 0.55 | 0.90 | 0.403 |
| 50 | 0.52 | 0.84 | 0.403 |
| 100 | 0.49 | 0.72 | 0.403 |

*Table: Precision@K on the client‑held‑out test set. The base rate (random pick) is 0.403. The Random Forest achieves a ~1.6× lift at K=50.*

**Permutation importance** showed that `impressions_prev` had a slightly higher importance (0.0966) than `avg_position_prev` (0.0261), but both are modest – the model relies on a combination of both signals rather than a single dominant feature. About 20‑25% of the top 20 logistic regression picks were false positives, and these tended to have lower impression volumes, reinforcing the need for a volume floor.

## 5. Limitations

*What this work cannot claim.*

- **Observational, not causal:** We measure association, not causation. A high predicted decline does not imply that changing the title *will* improve CTR.
- **Single month:** The model was trained and validated only on March 2026 data. Performance on other months is unknown.
- **Two features only:** The model does not capture content quality, query intent, or SERP features.
- **Client‑grouped split:** While it prevents client leakage, it may still not generalise to new clients.
- **Volume floor:** Predictions are only reliable for pages with at least 100 impressions.

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

The model outputs a ranked queue with three action levels:

- **Review title + content:** decline_prob ≥ 0.8 and impressions_prev ≥ 1000 (0 pages in this dataset)
- **Review title snippet:** decline_prob ≥ 0.6 and impressions_prev ≥ 500 (23,962 pages)
- **Monitor next cycle:** decline_prob ≥ 0.5 and impressions_prev ≥ 100 (13,499 pages)
- **No action:** remaining (39,939 pages)

**Reason codes** are concatenated flags: e.g., `high_decline_prob_high_visibility_good_position`. The queue is intended for **decision support**; a human editor must verify the page before acting.

**No‑go items:** Do not automate rewriting; do not use for pages with fewer than 100 impressions; do not treat as causal proof.

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

The paper includes the following artifacts:

1. **Precision@K table** – shown in Section 4 above.
2. **Distribution of decline probabilities** – histogram generated by `w07_action_playbook.ipynb` and available at `work/figures/decline_prob_distribution.png`.
3. **Ranked action queue** – exported to `work/outputs/action_queue.csv` by `w07_action_playbook.ipynb`.

These files are regenerated by the weekly notebooks and are referenced in the deployed paper at [https://syeddaniyalg.github.io/flyrank-work/](https://syeddaniyalg.github.io/flyrank-work/).

## 8. Demo Outline + Shareable Cuts

### 5-Minute Demo Outline

**1. The problem (1 min)**
- FlyRank's content decays over time.
- Editors have limited capacity – they need to know which pages to review FIRST.
- My lane: CTR / Engagement Opportunity Scoring – flag pages that rank well but get fewer clicks than peers at the same position.

**2. The data (30 sec)**
- FlyRank internship warehouse, March 2026.
- Filtered to 3.6M daily rows with GSC data.
- Aggregated to 77,400 content items with ≥100 impressions and a matching second‑half row.

**3. The method (1 min)**
- Features: impressions and average position from first 15 days.
- Label: CTR decline between first and second halves of March.
- Model: Random Forest (300 trees, max depth 6).
- Honest split: client‑grouped (no client appears in both train and test).
- Baseline: hand‑written rule (good tier × positive CTR gap × volume).

**4. The results (1 min)**
- Baseline Precision@50: 0.52
- Random Forest Precision@50: 0.84
- Lift: ~1.62× at K=50, ~1.64× at K=20
- Leakage test: adding ctr_prev and ctr_curr jumped Precision to 1.00 – confirming they are unsafe.

**5. The recommendation (1.5 min)**
- Ranked queue with actions:
  - Review title snippet: 23,962 pages (decline_prob ≥ 0.6, impressions ≥ 500)
  - Monitor next cycle: 13,499 pages (decline_prob ≥ 0.5, impressions ≥ 100)
  - No action: 39,939 pages
- Human review required – no automation.
- Reason codes explain each recommendation.

**Chart to show:** Distribution of predicted decline probabilities (histogram from work/figures/decline_prob_distribution.png)

---

### Shareable Cuts

#### Social post (methodology focus)

Built a Random Forest model to predict CTR decline for 77,400 content pages using 3.6M daily search records. Two features (impressions + avg position), client‑grouped validation, Precision@50 of 0.84 (baseline 0.52). No causality claims – just a ranked queue for editorial triage. Code and paper: https://github.com/syeddaniyalg/flyrank-work

#### Employer-facing summary (3 sentences)

I built a predictive ranking model that helps content editors prioritise pages likely to lose click‑through rate. Using 3.6 million daily search performance records from a production warehouse, the model achieves a Precision@50 of 0.84, compared to a hand‑written baseline of 0.52 – a ~1.6× lift. The output is a transparent action queue with reason codes, intended for human review, not automated rewriting.

#### Short portfolio blurb

CTR Decline Prediction: A decision‑support model for content review. Built on 3.6M daily search records, validated on client‑held‑out data, achieving 0.84 Precision@50. The model flags pages likely to decline, ranks them for editorial review, and provides transparent reason codes. Honest, observable, directional – no causal claims. Live paper: https://syeddaniyalg.github.io/flyrank-work/

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [x] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [x] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
